# Notebook 09 - Real Auckland GTFS-Realtime Validation

This notebook validates the real Auckland GTFS-Realtime delay pipeline for the MSE907 capstone project. It is separate from the Kaggle prototype notebooks and uses local Auckland GTFS-Realtime records, GTFS Static route metadata, and Open-Meteo hourly weather data.

The notebook is designed as a reproducible validation record. It can be rerun from the daily GTFS-Realtime files to regenerate the dashboard-ready outputs, while the export switch is kept explicit so future updates are traceable in GitHub.


## Validation Flow

1. Load repaired GTFS-Realtime daily files when available, with a fallback to repaired log files.
2. Clean headers, duplicate header rows, timestamps, booleans, and numeric delay fields.
3. Filter unreasonable delay outliers.
4. Create time features and delay-risk categories.
5. Merge route metadata from GTFS Static and validate route match rate.
6. Fetch hourly Open-Meteo weather and align it to GTFS UTC timestamps.
7. Generate operational recommendations and dashboard-ready tables in memory.
8. Regenerate dashboard-ready CSV outputs only from the documented input files and record the resulting Git changes.


In [1]:
# Core imports and project initialization

from pathlib import Path

import numpy as np
import pandas as pd
import json
from urllib.parse import urlencode
from urllib.request import urlopen

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
GTFS_REALTIME_DIR = DATA_RAW / "gtfs_realtime"
GTFS_DAILY_DIR = GTFS_REALTIME_DIR / "daily"
GTFS_STATIC_DIR = DATA_RAW / "gtfs_static"
SUMMARY_DIR = DATA_PROCESSED / "summaries"
PARQUET_GTFS_PATH = DATA_PROCESSED / "gtfs_realtime_cleaned.parquet"

print("Project root:", PROJECT_ROOT)
print("GTFS-Realtime folder exists:", GTFS_REALTIME_DIR.exists())
print("Daily GTFS-Realtime folder exists:", GTFS_DAILY_DIR.exists())
print("GTFS Static folder exists:", GTFS_STATIC_DIR.exists())


Project root: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport
GTFS-Realtime folder exists: True
Daily GTFS-Realtime folder exists: True
GTFS Static folder exists: True


## Step 1 - Locate Real Auckland GTFS-Realtime Inputs

The preferred source is the repaired daily GTFS-Realtime folder because it represents the ongoing real Auckland feed collection. The validation log remains available as a fallback.


In [21]:
# Locate candidate input files without changing them

validation_log_path = GTFS_REALTIME_DIR / "gtfs_realtime_log_validation.csv"
repaired_log_path = GTFS_REALTIME_DIR / "gtfs_realtime_log.csv"
daily_files = sorted(GTFS_DAILY_DIR.glob("gtfs_realtime_*.csv")) if GTFS_DAILY_DIR.exists() else []

print("Cleaned Parquet exists:", PARQUET_GTFS_PATH.exists())
if PARQUET_GTFS_PATH.exists():
    print("Cleaned Parquet size MB:", round(PARQUET_GTFS_PATH.stat().st_size / 1024 / 1024, 2))

print("Daily files found:", len(daily_files))
if daily_files:
    print("First daily file:", daily_files[0].name)
    print("Last daily file:", daily_files[-1].name)

print("Validation log exists:", validation_log_path.exists())
print("Repaired full log exists:", repaired_log_path.exists())


Cleaned Parquet exists: True
Cleaned Parquet size MB: 28.47
Daily files found: 23
First daily file: gtfs_realtime_2026-05-06.csv
Last daily file: gtfs_realtime_2026-05-28.csv
Validation log exists: True
Repaired full log exists: True


In [22]:
# Input selection
# Use "auto" for normal validation: Parquet first when readable, daily CSV fallback otherwise.

INPUT_MODE = "auto"  # options: "auto", "parquet", "daily", "validation_log", "repaired_log"
MAX_DAILY_FILES = None  # set to a small integer for a quick trial run; None uses all daily files

if INPUT_MODE == "auto":
    selected_gtfs_files = daily_files[:MAX_DAILY_FILES] if MAX_DAILY_FILES else daily_files
    requested_input_source = "parquet_if_available_else_daily"
elif INPUT_MODE == "parquet":
    selected_gtfs_files = []
    requested_input_source = "parquet_required"
elif INPUT_MODE == "daily":
    selected_gtfs_files = daily_files[:MAX_DAILY_FILES] if MAX_DAILY_FILES else daily_files
    requested_input_source = "daily_csv"
elif INPUT_MODE == "validation_log":
    selected_gtfs_files = [validation_log_path]
    requested_input_source = "validation_log_csv"
elif INPUT_MODE == "repaired_log":
    selected_gtfs_files = [repaired_log_path]
    requested_input_source = "repaired_log_csv"
else:
    raise ValueError("INPUT_MODE must be 'auto', 'parquet', 'daily', 'validation_log', or 'repaired_log'.")

if INPUT_MODE != "parquet" and not selected_gtfs_files:
    raise FileNotFoundError("No GTFS-Realtime CSV input files were selected. Check data/raw/gtfs_realtime.")

print("Selected input mode:", INPUT_MODE)
print("Requested input source:", requested_input_source)
print("Selected CSV fallback files:", len(selected_gtfs_files))
print("CSV fallback preview:", [path.name for path in selected_gtfs_files[:5]])


Selected input mode: auto
Requested input source: parquet_if_available_else_daily
Selected CSV fallback files: 23
CSV fallback preview: ['gtfs_realtime_2026-05-06.csv', 'gtfs_realtime_2026-05-07.csv', 'gtfs_realtime_2026-05-08.csv', 'gtfs_realtime_2026-05-09.csv', 'gtfs_realtime_2026-05-10.csv']


### Sanity Check - Input Discovery

Expected result: the cleaned Parquet file is detected when available, and daily GTFS-Realtime CSV files are also listed as fallback evidence. If Parquet cannot be read in the current environment, the notebook should clearly report the fallback reason and continue with daily CSV files.


## Step 2 - Load and Clean GTFS-Realtime Records

This step standardizes column names, removes duplicate header rows, converts delay values to numbers, converts timestamps to UTC datetimes, and keeps only records with usable delay and collection time values.


In [23]:
# Safe GTFS-Realtime loader

REQUIRED_GTFS_COLUMNS = [
    "collection_time_utc", "entity_id", "trip_id", "route_id", "direction_id",
    "start_time", "start_date", "timestamp", "delay_seconds", "is_deleted", "delay_minutes",
]

PARQUET_EXTRA_COLUMNS = ["collection_date", "source_file"]

def clean_column_names(frame):
    frame = frame.copy()
    frame.columns = frame.columns.astype(str).str.strip().str.lower().str.replace(" ", "_", regex=False)
    return frame

def remove_duplicate_header_rows(frame):
    frame = frame.copy()
    for column in frame.columns:
        frame = frame[frame[column].astype(str).str.strip().str.lower() != column]
    return frame

def parse_utc_datetime(series):
    """Parse UTC timestamps consistently across pandas versions used in project environments."""
    try:
        parsed = pd.to_datetime(series, errors="coerce", utc=True, format="mixed")
    except TypeError:
        parsed = pd.to_datetime(series, errors="coerce", utc=True)

    if parsed.notna().sum() == 0 and series.notna().sum() > 0:
        parsed = pd.to_datetime(series, errors="coerce", utc=True)

    return parsed

def validate_required_columns(frame, required_columns, label):
    missing_columns = sorted(set(required_columns) - set(frame.columns))
    if missing_columns:
        raise ValueError(f"Missing required {label} columns: {missing_columns}")

def normalize_realtime_types(frame):
    frame = frame.copy()

    for column in ["direction_id", "timestamp", "delay_seconds", "delay_minutes"]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    frame["collection_time_utc"] = parse_utc_datetime(frame["collection_time_utc"])

    if "is_deleted" in frame.columns:
        if frame["is_deleted"].dtype != bool:
            frame["is_deleted"] = (
                frame["is_deleted"]
                .astype(str)
                .str.strip()
                .str.upper()
                .map({"TRUE": True, "FALSE": False})
            )

    if "collection_date" not in frame.columns:
        frame["collection_date"] = frame["collection_time_utc"].dt.date.astype(str)

    return frame

def load_single_gtfs_file(path):
    frame = pd.read_csv(path, dtype=str, low_memory=False)
    frame = clean_column_names(frame)
    frame = remove_duplicate_header_rows(frame)
    frame["source_file"] = path.name
    return frame

def load_gtfs_realtime_files(paths):
    frames = [load_single_gtfs_file(path) for path in paths]
    combined = pd.concat(frames, ignore_index=True)
    validate_required_columns(combined, REQUIRED_GTFS_COLUMNS, "GTFS-Realtime CSV")

    combined = combined[REQUIRED_GTFS_COLUMNS + ["source_file"]].copy()
    combined = normalize_realtime_types(combined)

    before_drop = len(combined)
    combined = combined.dropna(subset=["collection_time_utc", "delay_minutes", "delay_seconds", "route_id", "trip_id"]).copy()
    return combined, before_drop - len(combined), "daily_csv_fallback"

def load_gtfs_realtime_parquet(path):
    frame = pd.read_parquet(path)
    frame = clean_column_names(frame)
    validate_required_columns(frame, REQUIRED_GTFS_COLUMNS, "GTFS-Realtime Parquet")

    if "source_file" not in frame.columns:
        frame["source_file"] = path.name

    keep_columns = REQUIRED_GTFS_COLUMNS + [column for column in PARQUET_EXTRA_COLUMNS if column in frame.columns]
    frame = frame[keep_columns].copy()
    frame = normalize_realtime_types(frame)

    before_drop = len(frame)
    frame = frame.dropna(subset=["collection_time_utc", "delay_minutes", "delay_seconds", "route_id", "trip_id"]).copy()
    return frame, before_drop - len(frame), "cleaned_parquet"

def load_gtfs_realtime_dataset(input_mode, parquet_path, csv_paths):
    parquet_error = None

    if input_mode in ["auto", "parquet"]:
        if parquet_path.exists():
            try:
                return load_gtfs_realtime_parquet(parquet_path)
            except Exception as error:
                parquet_error = error
                if input_mode == "parquet":
                    raise
                print("Parquet load unavailable; falling back to CSV daily/log input.")
                print("Parquet fallback reason:", type(error).__name__, str(error))
        elif input_mode == "parquet":
            raise FileNotFoundError(f"Required Parquet file not found: {parquet_path}")
        else:
            print("Cleaned Parquet not found; falling back to CSV daily/log input.")

    if not csv_paths:
        raise FileNotFoundError("No CSV fallback files are available for GTFS-Realtime loading.")

    return load_gtfs_realtime_files(csv_paths)

gtfs_df, dropped_invalid_rows, input_source_used = load_gtfs_realtime_dataset(
    INPUT_MODE,
    PARQUET_GTFS_PATH,
    selected_gtfs_files,
)

print("Input source used:", input_source_used)
print("Loaded GTFS-Realtime shape:", gtfs_df.shape)
print("Rows dropped during basic cleaning:", dropped_invalid_rows)
print("Source files represented:", gtfs_df["source_file"].nunique())
print("Time range:", gtfs_df["collection_time_utc"].min(), "to", gtfs_df["collection_time_utc"].max())
print("delay_minutes numeric:", pd.api.types.is_numeric_dtype(gtfs_df["delay_minutes"]))
print("collection_time_utc datetime:", pd.api.types.is_datetime64_any_dtype(gtfs_df["collection_time_utc"]))

gtfs_df.head()


Input source used: cleaned_parquet
Loaded GTFS-Realtime shape: (2530712, 13)
Rows dropped during basic cleaning: 0
Source files represented: 22
Time range: 2026-05-06 09:34:55+00:00 to 2026-05-27 06:43:26+00:00
delay_minutes numeric: True
collection_time_utc datetime: True


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv


In [24]:
# Basic GTFS-Realtime validation checks

print("Missing values by column:")
print(gtfs_df.isna().sum())

print("\nUnique routes:", gtfs_df["route_id"].nunique())
print("Unique trips:", gtfs_df["trip_id"].nunique())
print("Unique collection timestamps:", gtfs_df["collection_time_utc"].nunique())

print("\nDelay minutes summary:")
print(gtfs_df["delay_minutes"].describe())

print("\nDuplicate rows:", gtfs_df.duplicated().sum())


Missing values by column:
collection_time_utc    0
entity_id              0
trip_id                0
route_id               0
direction_id           0
start_time             0
start_date             0
timestamp              0
delay_seconds          0
is_deleted             0
delay_minutes          0
collection_date        0
source_file            0
dtype: int64

Unique routes: 522
Unique trips: 33720
Unique collection timestamps: 2246

Delay minutes summary:
count    2.530712e+06
mean    -4.387684e-01
std      1.098973e+01
min     -1.039567e+03
25%     -2.950000e+00
50%     -1.666667e-02
75%      1.500000e+00
max      7.593833e+02
Name: delay_minutes, dtype: float64

Duplicate rows: 0


### Sanity Check - GTFS-Realtime Cleaning

Expected result: `delay_seconds` and `delay_minutes` are numeric, `collection_time_utc` is timezone-aware UTC, duplicate header rows are gone, and the remaining records represent real Auckland routes and trips.


## Step 3 - Filter Delay Outliers

Very large negative or positive delays can distort validation summaries. This notebook keeps delays from -60 to +120 minutes. Those bounds are intentionally visible so they can be defended or adjusted in the capstone report.


In [40]:
# Outlier handling

MIN_REASONABLE_DELAY_MINUTES = -60
MAX_REASONABLE_DELAY_MINUTES = 120

outlier_mask = gtfs_df["delay_minutes"].between(MIN_REASONABLE_DELAY_MINUTES, MAX_REASONABLE_DELAY_MINUTES, inclusive="both")
gtfs_clean = gtfs_df.loc[outlier_mask].copy()
outlier_rows_removed = len(gtfs_df) - len(gtfs_clean)

print("Original rows:", len(gtfs_df))
print("Rows after outlier filter:", len(gtfs_clean))
print("Outlier rows removed:", outlier_rows_removed)
print("Outlier removal rate:", round(outlier_rows_removed / len(gtfs_df) * 100, 2), "%")

print("\nCleaned delay summary:")
print(gtfs_clean["delay_minutes"].describe())


Original rows: 2530712
Rows after outlier filter: 2529972
Outlier rows removed: 740
Outlier removal rate: 0.03 %

Cleaned delay summary:
count    2.529972e+06
mean    -4.192677e-01
std      5.403794e+00
min     -5.996667e+01
25%     -2.950000e+00
50%     -1.666667e-02
75%      1.500000e+00
max      1.199500e+02
Name: delay_minutes, dtype: float64


### Sanity Check - Outlier Filter

Expected result: only a small share of records should be removed. If many records are removed, the raw feed or delay calculation should be reviewed before using the data for decision support.


## Step 4 - Create Temporal Features

Temporal features make the validation useful for operations. They allow the dashboard and summaries to compare delay patterns by hour, weekday, and day of month.


In [43]:
# Temporal feature engineering

gtfs_clean["trip_hour"] = gtfs_clean["collection_time_utc"].dt.hour
gtfs_clean["weekday"] = gtfs_clean["collection_time_utc"].dt.dayofweek
gtfs_clean["day_of_month"] = gtfs_clean["collection_time_utc"].dt.day
gtfs_clean["collection_date"] = gtfs_clean["collection_time_utc"].dt.date
gtfs_clean["delay_abs_minutes"] = gtfs_clean["delay_minutes"].abs()

print("Feature-engineered shape:", gtfs_clean.shape)
print("Trip hour range:", gtfs_clean["trip_hour"].min(), "to", gtfs_clean["trip_hour"].max())
print("Weekdays represented:", sorted(gtfs_clean["weekday"].dropna().unique().tolist()))

gtfs_clean[["collection_time_utc", "collection_date", "route_id", "trip_id", "delay_minutes", "trip_hour", "weekday", "day_of_month"]].head()


Feature-engineered shape: (2529972, 17)
Trip hour range: 0 to 23
Weekdays represented: [0, 1, 2, 3, 4, 5, 6]


,collection_time_utc,collection_date,route_id,trip_id,delay_minutes,trip_hour,weekday,day_of_month
0,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,902-98011-82800-1-cdd3d9f0,0.000000,9,2,6
1,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,902-98011-88200-1-cdd3d9f0,0.000000,9,2,6
2,2026-05-06 09:34:55+00:00,2026-05-06,EAST-201,246-850001-75660-2-5270112-3da0c47e,0.833333,9,2,6
3,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,248-840042-73620-2-6159400-f7b76986,0.283333,9,2,6
4,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,248-840085-78780-2-6164400-25f91d7f,0.000000,9,2,6


### Sanity Check - Temporal Features

Expected result: `trip_hour`, `weekday`, and `day_of_month` are populated for every retained GTFS-Realtime record.


## Step 5 - Merge GTFS Static Route Metadata

GTFS Static route metadata adds route names, route types, and agency details. This makes the realtime records easier to interpret and supports route-level dashboard views.


In [57]:
# Load GTFS Static route metadata

routes_path = GTFS_STATIC_DIR / "routes.txt"
if not routes_path.exists():
    raise FileNotFoundError(f"GTFS Static routes file not found: {routes_path}")

routes_df = pd.read_csv(routes_path, dtype=str)
routes_df = clean_column_names(routes_df)

required_route_columns = ["route_id", "route_short_name", "route_long_name", "route_type", "agency_id"]
missing_route_columns = sorted(set(required_route_columns) - set(routes_df.columns))
if missing_route_columns:
    raise ValueError(f"Missing GTFS Static route columns: {missing_route_columns}")

routes_lookup = routes_df[required_route_columns].drop_duplicates(subset=["route_id"]).copy()

print("Routes shape:", routes_df.shape)
print("Route lookup shape:", routes_lookup.shape)
routes_lookup.head()


Routes shape: (220, 11)
Route lookup shape: (220, 5)


,route_id,route_short_name,route_long_name,route_type,agency_id
0,101-202,101,101,3,NZB
1,STH-201,STH,STH,2,AM
2,TIRI-240,TIRI,TIRI,4,EXPNZ
3,TMK-202,TMK,TMK,3,NZB
4,WEST-201,WEST,WEST,2,AM


In [61]:
# Merge realtime records with GTFS Static route metadata

gtfs_routes = gtfs_clean.merge(routes_lookup, on="route_id", how="left", validate="many_to_one")

matched_routes = gtfs_routes["route_short_name"].notna().sum()
total_route_records = len(gtfs_routes)
route_match_rate = matched_routes / total_route_records if total_route_records else np.nan

print("Realtime GTFS before route merge:", gtfs_clean.shape)
print("Realtime GTFS after route merge:", gtfs_routes.shape)
print("Matched route records:", matched_routes)
print("Route match rate:", round(route_match_rate * 100, 2), "%")

print("\nMissing route metadata:")
print(gtfs_routes[["route_short_name", "route_long_name", "route_type", "agency_id"]].isna().sum())

gtfs_routes.head()


Realtime GTFS before route merge: (2529972, 17)
Realtime GTFS after route merge: (2529972, 21)
Matched route records: 2465383
Route match rate: 97.45 %

Missing route metadata:
route_short_name    64589
route_long_name     64589
route_type          64589
agency_id           64589
dtype: int64


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file,trip_hour,weekday,day_of_month,delay_abs_minutes,route_short_name,route_long_name,route_type,agency_id
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,MTIA,MTIA,4,FGL
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,MTIA,MTIA,4,FGL
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.833333,EAST,EAST,2,AM
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.283333,ONE,ONE,2,AM
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,ONE,ONE,2,AM


### Sanity Check - Route Metadata

Expected result: the route match rate should be high. Any unmatched route IDs should be reviewed because they may indicate stale static GTFS files, realtime-only route identifiers, or feed collection issues.


## Step 5A - Route Coverage Limitation Check

The GTFS Static merge can legitimately miss some realtime route identifiers when the realtime feed contains school, special, or operational variants that are not present in the static `routes.txt` snapshot. This section quantifies those unmatched records so the route match rate can be reported transparently.


In [73]:
# Inspect unmatched realtime route IDs after the GTFS Static merge

unmatched_routes = gtfs_routes[gtfs_routes["route_short_name"].isna()].copy()

unmatched_route_summary = (
    unmatched_routes
    .groupby("route_id", as_index=False)
    .agg(
        records=("trip_id", "size"),
        unique_trips=("trip_id", "nunique"),
        avg_delay_minutes=("delay_minutes", "mean"),
        max_delay_minutes=("delay_minutes", "max"),
    )
    .sort_values(["records", "avg_delay_minutes"], ascending=[False, False])
)

unmatched_route_summary["avg_delay_minutes"] = unmatched_route_summary["avg_delay_minutes"].round(2)
unmatched_route_summary["route_prefix"] = unmatched_route_summary["route_id"].str.extract(r"^([A-Za-z]+)", expand=False).fillna("numeric_or_other")

unmatched_by_source_file = (
    unmatched_routes
    .groupby("source_file", as_index=False)
    .size()
    .rename(columns={"size": "unmatched_records"})
)

print("Unmatched records:", len(unmatched_routes))
print("Unmatched unique routes:", unmatched_routes["route_id"].nunique())
print("Unmatched share:", round(len(unmatched_routes) / len(gtfs_routes) * 100, 2), "%")

print("\nUnmatched route prefix counts:")
print(unmatched_route_summary["route_prefix"].value_counts())

print("\nTop unmatched routes:")
unmatched_route_summary.head(20)


Unmatched records: 64589
Unmatched unique routes: 316
Unmatched share: 2.55 %

Unmatched route prefix counts:
S    316
Name: route_prefix, dtype: int64

Top unmatched routes:


,route_id,records,unique_trips,avg_delay_minutes,max_delay_minutes,route_prefix
286,S510-202,690,5,-1.94,12.783333,S
311,S547-202,646,5,-0.45,8.983333,S
58,S017F-203,628,6,-2.27,14.116667,S
137,S050B-203,587,5,-2.68,35.750000,S
234,S413-202,545,4,-0.47,10.616667,S
18,S006C-203,529,4,-0.62,41.700000,S
238,S417-202,527,4,1.48,8.066667,S
21,S007F-203,480,8,2.46,17.883333,S
306,S542-202,470,4,-1.17,14.850000,S
39,S013-203,444,3,0.96,14.000000,S


### Sanity Check - Unmatched Routes

Expected result: unmatched records should be a small share of the full realtime dataset. If most unmatched route IDs share a prefix such as `S`, document them as static coverage limitations and avoid treating them as a failed route merge.


## Step 6 - Create Delay-Risk Categories

Delay-risk categories translate numeric delay values into operational severity levels. This keeps the decision engine understandable for non-technical readers.


In [81]:
# Delay-risk category logic

RISK_ORDER = ["Low", "Medium", "High", "Severe"]

def classify_realtime_delay_risk(delay_minutes):
    if delay_minutes <= 5:
        return "Low"
    if delay_minutes <= 15:
        return "Medium"
    if delay_minutes <= 25:
        return "High"
    return "Severe"

gtfs_routes["delay_risk"] = gtfs_routes["delay_abs_minutes"].apply(classify_realtime_delay_risk)
gtfs_routes["delay_risk"] = pd.Categorical(gtfs_routes["delay_risk"], categories=RISK_ORDER, ordered=True)

risk_summary = gtfs_routes["delay_risk"].value_counts(sort=False).rename_axis("delay_risk").reset_index(name="records")
risk_summary["share_pct"] = (risk_summary["records"] / len(gtfs_routes) * 100).round(2)
risk_summary


,delay_risk,records,share_pct
0,Low,1965777,77.70
1,Medium,525781,20.78
2,High,28826,1.14
3,Severe,9588,0.38


### Sanity Check - Delay Risk

Expected result: the risk table should include all four categories. If the severe category is large, inspect whether the route, collection period, or outlier bounds explain the pattern.


## Step 7 - Fetch Open-Meteo Hourly Weather

Weather is aligned to GTFS collection timestamps by hour. The request uses UTC so the weather timestamps match `collection_time_utc` directly. If network access is unavailable during reruns, use a documented local Open-Meteo extract with the same hourly columns and date range instead of calling the API.


In [91]:
# Open-Meteo hourly weather request

latitude = -36.8485
longitude = 174.7633

weather_start_date = gtfs_routes["collection_time_utc"].min().strftime("%Y-%m-%d")
weather_end_date = gtfs_routes["collection_time_utc"].max().strftime("%Y-%m-%d")

weather_url = "https://archive-api.open-meteo.com/v1/archive"
weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "start_date": weather_start_date,
    "end_date": weather_end_date,
    "hourly": ",".join([
        "temperature_2m",
        "precipitation",
        "rain",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]),
    "timezone": "UTC",
}

print("Weather request date range:", weather_start_date, "to", weather_end_date)

weather_json = None
weather_df = pd.DataFrame()
weather_source = "Open-Meteo API"

try:
    weather_request_url = f"{weather_url}?{urlencode(weather_params)}"
    with urlopen(weather_request_url, timeout=30) as response:
        weather_status = response.status
        weather_json = json.loads(response.read().decode("utf-8"))

    print("Weather API status:", weather_status)

    if "hourly" not in weather_json:
        raise ValueError("Open-Meteo response does not contain an hourly weather table.")

except Exception as error:
    weather_source = "local decision_engine_output.csv fallback"
    print("Weather API unavailable; using local weather fallback.")
    print("Weather fallback reason:", type(error).__name__, str(error))

    local_weather_path = DATA_PROCESSED / "decision_engine_output.csv"
    local_weather_columns = [
        "collection_time_utc",
        "temperature_2m",
        "precipitation",
        "rain",
        "relative_humidity_2m",
        "wind_speed_10m",
    ]

    if not local_weather_path.exists():
        raise FileNotFoundError(
            "Open-Meteo API was unavailable and local decision_engine_output.csv was not found."
        )

    weather_df = pd.read_csv(
        local_weather_path,
        usecols=local_weather_columns,
        low_memory=False,
    )
    weather_df["weather_hour"] = parse_utc_datetime(weather_df["collection_time_utc"]).dt.floor("h")
    weather_df = (
        weather_df
        .dropna(subset=["weather_hour"])
        .drop_duplicates(subset=["weather_hour"])
        [[
            "weather_hour",
            "temperature_2m",
            "precipitation",
            "rain",
            "relative_humidity_2m",
            "wind_speed_10m",
        ]]
        .copy()
    )

print("Weather source:", weather_source)


Weather request date range: 2026-05-06 to 2026-05-27
Weather API status: 200
Weather source: Open-Meteo API


In [97]:
# Convert weather response to dataframe

if weather_df.empty:
    weather_df = pd.DataFrame(weather_json["hourly"])
    weather_df = clean_column_names(weather_df)

    if "time" not in weather_df.columns:
        raise ValueError("Weather data is missing the time column.")

    weather_df["weather_hour"] = parse_utc_datetime(weather_df["time"])
else:
    weather_df = clean_column_names(weather_df)

weather_df = (
    weather_df
    .dropna(subset=["weather_hour"])
    .drop_duplicates(subset=["weather_hour"])
    .copy()
)

print("Weather dataset shape:", weather_df.shape)
print("Weather time range:", weather_df["weather_hour"].min(), "to", weather_df["weather_hour"].max())
weather_df.head()


Weather dataset shape: (528, 7)
Weather time range: 2026-05-06 00:00:00+00:00 to 2026-05-27 23:00:00+00:00


,time,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m,weather_hour
0,2026-05-06T00:00,18.4,0.2,0.2,83,12.9,2026-05-06 00:00:00+00:00
1,2026-05-06T01:00,19.0,0.0,0.0,79,13.5,2026-05-06 01:00:00+00:00
2,2026-05-06T02:00,19.3,0.1,0.1,76,13.9,2026-05-06 02:00:00+00:00
3,2026-05-06T03:00,19.2,0.0,0.0,75,13.9,2026-05-06 03:00:00+00:00
4,2026-05-06T04:00,18.8,0.0,0.0,77,14.4,2026-05-06 04:00:00+00:00


In [105]:
# Align GTFS timestamps to hourly weather timestamps

gtfs_routes["weather_hour"] = gtfs_routes["collection_time_utc"].dt.floor("h")

print("Realtime weather-hour range:", gtfs_routes["weather_hour"].min(), "to", gtfs_routes["weather_hour"].max())
print("Weather table range:", weather_df["weather_hour"].min(), "to", weather_df["weather_hour"].max())

print("\nGTFS hourly timestamp sample:")
print(gtfs_routes["weather_hour"].head())

print("\nWeather hourly timestamp sample:")
print(weather_df["weather_hour"].head())


Realtime weather-hour range: 2026-05-06 09:00:00+00:00 to 2026-05-27 06:00:00+00:00
Weather table range: 2026-05-06 00:00:00+00:00 to 2026-05-27 23:00:00+00:00

GTFS hourly timestamp sample:
0   2026-05-06 09:00:00+00:00
1   2026-05-06 09:00:00+00:00
2   2026-05-06 09:00:00+00:00
3   2026-05-06 09:00:00+00:00
4   2026-05-06 09:00:00+00:00
Name: weather_hour, dtype: datetime64[ns, UTC]

Weather hourly timestamp sample:
0   2026-05-06 00:00:00+00:00
1   2026-05-06 01:00:00+00:00
2   2026-05-06 02:00:00+00:00
3   2026-05-06 03:00:00+00:00
4   2026-05-06 04:00:00+00:00
Name: weather_hour, dtype: datetime64[ns, UTC]


In [112]:
# Merge realtime GTFS records with hourly weather data

weather_columns = ["weather_hour", "temperature_2m", "precipitation", "rain", "relative_humidity_2m", "wind_speed_10m"]
weather_value_columns = [column for column in weather_columns if column != "weather_hour"]
missing_weather_columns = sorted(set(weather_columns) - set(weather_df.columns))
if missing_weather_columns:
    raise ValueError(f"Missing weather columns: {missing_weather_columns}")

weather_lookup = (
    weather_df[weather_columns]
    .drop_duplicates(subset=["weather_hour"])
    .set_index("weather_hour")
)

# Use hourly lookup mapping instead of a full dataframe merge to reduce memory use on multi-million-row datasets.
gtfs_weather = gtfs_routes.copy(deep=False)
for column in weather_value_columns:
    gtfs_weather[column] = gtfs_weather["weather_hour"].map(weather_lookup[column])
    gtfs_weather[column] = pd.to_numeric(gtfs_weather[column], errors="coerce", downcast="float")

matched_weather = gtfs_weather["temperature_2m"].notna().sum()
total_weather_records = len(gtfs_weather)
weather_match_rate = matched_weather / total_weather_records if total_weather_records else np.nan

print("Integrated GTFS + weather shape:", gtfs_weather.shape)
print("Weather matched records:", matched_weather)
print("Weather match rate:", round(weather_match_rate * 100, 2), "%")

print("\nMissing weather values:")
print(gtfs_weather[weather_value_columns].isna().sum())

gtfs_weather.head()


Integrated GTFS + weather shape: (2529972, 28)
Weather matched records: 2529972
Weather match rate: 100.0 %

Missing weather values:
temperature_2m          0
precipitation           0
rain                    0
relative_humidity_2m    0
wind_speed_10m          0
dtype: int64


,collection_time_utc,entity_id,trip_id,route_id,direction_id,start_time,start_date,timestamp,delay_seconds,is_deleted,delay_minutes,collection_date,source_file,trip_hour,weekday,day_of_month,delay_abs_minutes,route_short_name,route_long_name,route_type,agency_id,delay_risk,weather_hour,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m
0,2026-05-06 09:34:55+00:00,902-98011-82800-1-cdd3d9f0,902-98011-82800-1-cdd3d9f0,MTIA-209,0,23:00:00,20260506,1778000407,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,MTIA,MTIA,4,FGL,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
1,2026-05-06 09:34:55+00:00,902-98011-88200-1-cdd3d9f0,902-98011-88200-1-cdd3d9f0,MTIA-209,0,24:30:00,20260506,1778000413,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,MTIA,MTIA,4,FGL,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
2,2026-05-06 09:34:55+00:00,246-850001-75660-2-5270112-3da0c47e,246-850001-75660-2-5270112-3da0c47e,EAST-201,0,21:01:00,20260506,1778060030,50,False,0.833333,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.833333,EAST,EAST,2,AM,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
3,2026-05-06 09:34:55+00:00,248-840042-73620-2-6159400-f7b76986,248-840042-73620-2-6159400-f7b76986,ONE-201,1,20:27:00,20260506,1778057120,17,False,0.283333,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.283333,ONE,ONE,2,AM,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1
4,2026-05-06 09:34:55+00:00,248-840085-78780-2-6164400-25f91d7f,248-840085-78780-2-6164400-25f91d7f,ONE-201,0,21:53:00,20260506,1778051978,0,False,0.000000,2026-05-06,gtfs_realtime_2026-05-06.csv,9,2,6,0.000000,ONE,ONE,2,AM,Low,2026-05-06 09:00:00+00:00,17.4,0.0,0.0,86.0,13.1


### Sanity Check - Weather Integration

Expected result: the weather match rate should be close to 100%. If it is low, check timezone handling first, then confirm the Open-Meteo date range covers the GTFS-Realtime collection period.


## Step 8 - Operational Recommendations

The recommendation rules translate delay-risk categories into plain-language transport operations actions. These are decision-support recommendations, not automated control decisions.


In [115]:
# Operational recommendation logic

def generate_realtime_recommendation(risk_level):
    risk_level = str(risk_level)
    if risk_level == "Low":
        return "No operational action required"
    if risk_level == "Medium":
        return "Monitor route conditions"
    if risk_level == "High":
        return "Adjust service headway"
    return "Deploy standby bus or supervisor review"

gtfs_weather["recommended_action"] = gtfs_weather["delay_risk"].apply(generate_realtime_recommendation)

recommendation_summary = gtfs_weather["recommended_action"].value_counts().rename_axis("recommended_action").reset_index(name="records")
recommendation_summary["share_pct"] = (recommendation_summary["records"] / len(gtfs_weather) * 100).round(2)
recommendation_summary


,recommended_action,records,share_pct
0,No operational action required,1965777,77.70
1,Monitor route conditions,525781,20.78
2,Adjust service headway,28826,1.14
3,Deploy standby bus or supervisor review,9588,0.38


In [120]:
# Dashboard-ready decision engine table held in memory only

decision_engine_output = gtfs_weather[[
    "collection_time_utc", "collection_date", "route_id", "route_short_name", "route_long_name", "route_type", "agency_id",
    "trip_id", "direction_id", "delay_seconds", "delay_minutes", "delay_risk", "recommended_action", "trip_hour", "weekday",
    "day_of_month", "temperature_2m", "precipitation", "rain", "relative_humidity_2m", "wind_speed_10m", "source_file",
]].copy()

print("Decision engine output shape:", decision_engine_output.shape)
decision_engine_output.head()


Decision engine output shape: (2529972, 22)


,collection_time_utc,collection_date,route_id,route_short_name,route_long_name,route_type,agency_id,trip_id,direction_id,delay_seconds,delay_minutes,delay_risk,recommended_action,trip_hour,weekday,day_of_month,temperature_2m,precipitation,rain,relative_humidity_2m,wind_speed_10m,source_file
0,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,MTIA,MTIA,4,FGL,902-98011-82800-1-cdd3d9f0,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
1,2026-05-06 09:34:55+00:00,2026-05-06,MTIA-209,MTIA,MTIA,4,FGL,902-98011-88200-1-cdd3d9f0,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
2,2026-05-06 09:34:55+00:00,2026-05-06,EAST-201,EAST,EAST,2,AM,246-850001-75660-2-5270112-3da0c47e,0,50,0.833333,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
3,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,ONE,ONE,2,AM,248-840042-73620-2-6159400-f7b76986,1,17,0.283333,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv
4,2026-05-06 09:34:55+00:00,2026-05-06,ONE-201,ONE,ONE,2,AM,248-840085-78780-2-6164400-25f91d7f,0,0,0.000000,Low,No operational action required,9,2,6,17.4,0.0,0.0,86.0,13.1,gtfs_realtime_2026-05-06.csv


### Sanity Check - Recommendations

Expected result: every retained record should have a delay-risk category and a recommended action. Recommendations are intentionally simple so they can be explained in the final report and dashboard.


## Step 9 - Dashboard Summary Tables

These tables are prepared in memory for Streamlit/dashboard use. They are written to disk only when `WRITE_OUTPUTS` is deliberately set to `True` for a reproducible output refresh.


In [124]:
# Dashboard-ready summaries held in memory only

daily_summary = decision_engine_output.groupby("collection_date", as_index=False).agg(
    records=("trip_id", "size"),
    unique_routes=("route_id", "nunique"),
    unique_trips=("trip_id", "nunique"),
    avg_delay_minutes=("delay_minutes", "mean"),
    min_delay_minutes=("delay_minutes", "min"),
    max_delay_minutes=("delay_minutes", "max"),
    weather_match_records=("temperature_2m", lambda values: values.notna().sum()),
)
daily_summary["avg_delay_minutes"] = daily_summary["avg_delay_minutes"].round(2)

route_daily_summary = decision_engine_output.groupby(
    ["collection_date", "route_id", "route_short_name", "route_long_name"], dropna=False, as_index=False
).agg(
    records=("trip_id", "size"),
    unique_trips=("trip_id", "nunique"),
    avg_delay_minutes=("delay_minutes", "mean"),
    max_delay_minutes=("delay_minutes", "max"),
)
route_daily_summary["avg_delay_minutes"] = route_daily_summary["avg_delay_minutes"].round(2)

top_delayed_routes = decision_engine_output.groupby(["route_id", "route_short_name", "route_long_name"], dropna=False, as_index=False).agg(
    records=("trip_id", "size"),
    avg_delay_minutes=("delay_minutes", "mean"),
    max_delay_minutes=("delay_minutes", "max"),
).sort_values(["avg_delay_minutes", "records"], ascending=[False, False]).head(20)
top_delayed_routes["avg_delay_minutes"] = top_delayed_routes["avg_delay_minutes"].round(2)

print("Daily summary shape:", daily_summary.shape)
print("Route daily summary shape:", route_daily_summary.shape)
print("Top delayed routes shape:", top_delayed_routes.shape)
top_delayed_routes.head()


Daily summary shape: (22, 8)
Route daily summary shape: (8575, 8)
Top delayed routes shape: (20, 6)


,route_id,route_short_name,route_long_name,records,avg_delay_minutes,max_delay_minutes
186,HUIA-404,HUIA,HUIA,64,50.15,119.950000
473,S459-203,NaN,NaN,278,11.73,47.050000
451,S429-221,NaN,NaN,151,9.43,14.800000
326,S046D-221,NaN,NaN,145,9.07,32.883333
363,S061A-203,NaN,NaN,122,9.04,20.433333


In [127]:
# Intervention logic table held in memory only

intervention_logic = pd.DataFrame([
    {"delay_risk": "Low", "delay_minutes_rule": "0 to 5 minutes absolute delay", "recommended_action": "No operational action required", "operational_meaning": "Service is operating within an acceptable tolerance."},
    {"delay_risk": "Medium", "delay_minutes_rule": "More than 5 and up to 15 minutes absolute delay", "recommended_action": "Monitor route conditions", "operational_meaning": "Route should be watched for emerging disruption patterns."},
    {"delay_risk": "High", "delay_minutes_rule": "More than 15 and up to 25 minutes absolute delay", "recommended_action": "Adjust service headway", "operational_meaning": "Operations may need spacing or dispatch adjustments."},
    {"delay_risk": "Severe", "delay_minutes_rule": "More than 25 minutes absolute delay", "recommended_action": "Deploy standby bus or supervisor review", "operational_meaning": "High disruption risk that may need active intervention."},
])

intervention_logic


,delay_risk,delay_minutes_rule,recommended_action,operational_meaning
0,Low,0 to 5 minutes absolute delay,No operational action required,Service is operating within an acceptable tole...
1,Medium,More than 5 and up to 15 minutes absolute delay,Monitor route conditions,Route should be watched for emerging disruptio...
2,High,More than 15 and up to 25 minutes absolute delay,Adjust service headway,Operations may need spacing or dispatch adjust...
3,Severe,More than 25 minutes absolute delay,Deploy standby bus or supervisor review,High disruption risk that may need active inte...


### Sanity Check - Dashboard Tables

Expected result: the dashboard tables are generated in memory and use real Auckland route IDs. Dashboard files should be regenerated only after the validation checks above have passed and the changed outputs are reviewed in Git.


## Step 10 - Reproducible Export Configuration

The output paths below define the dashboard-ready files used by the later Streamlit and decision-support layers. Keep `WRITE_OUTPUTS = False` while inspecting or testing the notebook. Set it to `True` only when intentionally regenerating project outputs from the validated real Auckland pipeline.


In [ ]:
# Optional dashboard-ready exports
# Keep False for inspection/test runs. Set True only for a controlled output refresh.

WRITE_OUTPUTS = False

output_paths = {
    "realtime_with_routes_sample": DATA_INTERIM / "realtime_with_routes_sample.csv",
    "gtfs_weather_integrated_sample": DATA_INTERIM / "gtfs_weather_integrated_sample.csv",
    "decision_engine_output": DATA_PROCESSED / "decision_engine_output.csv",
    "intervention_logic": DATA_PROCESSED / "intervention_logic.csv",
    "daily_summary": SUMMARY_DIR / "gtfs_realtime_daily_summary.csv",
    "route_daily_summary": SUMMARY_DIR / "gtfs_realtime_route_daily_summary.csv",
    "top_delayed_routes": SUMMARY_DIR / "gtfs_realtime_top_delayed_routes.csv",
}

for name, path in output_paths.items():
    print(f"{name}: {path} | exists={path.exists()}")

if WRITE_OUTPUTS:
    DATA_INTERIM.mkdir(parents=True, exist_ok=True)
    SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

    gtfs_routes.head(10000).to_csv(output_paths["realtime_with_routes_sample"], index=False)
    gtfs_weather.head(10000).to_csv(output_paths["gtfs_weather_integrated_sample"], index=False)
    decision_engine_output.to_csv(output_paths["decision_engine_output"], index=False)
    intervention_logic.to_csv(output_paths["intervention_logic"], index=False)
    daily_summary.to_csv(output_paths["daily_summary"], index=False)
    route_daily_summary.to_csv(output_paths["route_daily_summary"], index=False)
    top_delayed_routes.to_csv(output_paths["top_delayed_routes"], index=False)

    print("Exports completed.")
else:
    print("WRITE_OUTPUTS is False. No files were written or overwritten.")


realtime_with_routes_sample: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\interim\realtime_with_routes_sample.csv | exists=True
gtfs_weather_integrated_sample: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\interim\gtfs_weather_integrated_sample.csv | exists=True
decision_engine_output: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\decision_engine_output.csv | exists=True
intervention_logic: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\intervention_logic.csv | exists=True
daily_summary: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\summaries\gtfs_realtime_daily_summary.csv | exists=True
route_daily_summary: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\summaries\gtfs_realtime_route_daily_summary.csv | exists=True
top_delayed_routes: D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\data\processed\summaries\gtfs_realtime_top_delaye

## Final Reproducibility Checklist

Before committing regenerated outputs, confirm the following:

- The selected GTFS-Realtime input is the real Auckland repaired daily feed or a clearly documented fallback.
- Delay fields are numeric and unreasonable outliers have been filtered.
- Route metadata match rate is acceptable.
- Weather match rate is acceptable after UTC hourly alignment.
- Decision recommendations are generated from real Auckland GTFS records, not Kaggle prototype rows.
- Exported CSV files are reviewed with `git status` so the committed outputs match this real Auckland validation run.
